# slidingwindow not extended

In [1]:
import os, glob, json
import numpy as np
import torch
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import matplotlib.pyplot as plt

def convert_to_serializable(obj):
    """JSON序列化转换"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    else:
        return obj

def compute_window_similarity(window_a, window_b, method='cosine'):
    """
    计算两个窗口的相似度
    
    Args:
        window_a, window_b: (window_size, feature_dim) 的特征窗口
        method: 'cosine', 'l2', 'correlation', 'kl_div'
    """
    if method == 'cosine':
        # 余弦相似度
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        
        norm_a = np.linalg.norm(mean_a)
        norm_b = np.linalg.norm(mean_b)
        
        if norm_a == 0 or norm_b == 0:
            return 0.0
        
        cosine_sim = np.dot(mean_a, mean_b) / (norm_a * norm_b)
        return float(cosine_sim)
    
    elif method == 'l2':
        # L2距离相似度 (距离越小越相似)
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        l2_dist = np.linalg.norm(mean_a - mean_b)
        # 转换为相似度 (0-1范围)
        similarity = np.exp(-l2_dist)
        return float(similarity)
    
    elif method == 'correlation':
        # 皮尔森相关系数
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        
        if np.std(mean_a) == 0 or np.std(mean_b) == 0:
            return 1.0 if np.allclose(mean_a, mean_b) else 0.0
        
        correlation = np.corrcoef(mean_a, mean_b)[0, 1]
        return float(correlation) if not np.isnan(correlation) else 0.0
    
    elif method == 'histogram':
        # 直方图相似度
        # 将特征量化为直方图
        hist_a, _ = np.histogram(window_a.flatten(), bins=50, density=True)
        hist_b, _ = np.histogram(window_b.flatten(), bins=50, density=True)
        
        # 计算直方图交集
        intersection = np.minimum(hist_a, hist_b).sum()
        return float(intersection)

def sliding_window_segmentation(feats, window_size=10, step_size=1, threshold=0.8, 
                               similarity_method='cosine', min_segment_frames=10):
    """
    滑动窗口视频分割算法
    
    Args:
        feats: (T, D) 视频特征矩阵
        window_size: 窗口大小（帧数）
        step_size: 滑动步长
        threshold: 相似度阈值，低于此值认为是边界
        similarity_method: 相似度计算方法
        min_segment_frames: 最小段长度（帧数）
    
    Returns:
        boundaries: 边界帧索引列表
        similarities: 相似度序列
    """
    T, D = feats.shape
    boundaries = []
    similarities = []
    positions = []
    
    print(f"    Sliding window: size={window_size}, step={step_size}, threshold={threshold}")
    
    # 滑动窗口检测
    pos = 0
    while pos + 2 * window_size <= T:
        # 定义两个相邻窗口
        window_a = feats[pos:pos + window_size]
        window_b = feats[pos + window_size:pos + 2 * window_size]
        
        # 计算相似度
        similarity = compute_window_similarity(window_a, window_b, similarity_method)
        similarities.append(similarity)
        positions.append(pos + window_size)  # 边界位置在两个窗口之间
        
        # 如果相似度低于阈值，认为是边界
        if similarity < threshold:
            boundary_pos = pos + window_size
            
            # 检查是否满足最小段长度约束
            if not boundaries or boundary_pos - boundaries[-1] >= min_segment_frames:
                boundaries.append(boundary_pos)
                print(f"      Boundary found at frame {boundary_pos}, similarity={similarity:.3f}")
        
        pos += step_size
    
    return boundaries, similarities, positions

def adaptive_sliding_window(feats, fps=10, debug=False):
    """
    自适应滑动窗口算法 - 快速版本用于大规模处理
    """
    if debug:
        print(f"  Adaptive sliding window on {feats.shape[0]} frames")
    
    # 简化的参数组合 - 减少计算量
    window_sizes = [10, 15]  # 只测试1s和1.5s
    thresholds = [0.8, 0.85]  # 只测试两个阈值
    methods = ['cosine']      # 只用最有效的余弦相似度
    
    best_boundaries = []
    best_score = 0
    best_params = {}
    
    for window_size in window_sizes:
        for threshold in thresholds:
            for method in methods:
                try:
                    # 最小段长度 = 1秒
                    min_segment_frames = max(fps, window_size)
                    
                    boundaries, similarities, positions = sliding_window_segmentation(
                        feats, window_size, step_size=2, threshold=threshold,  # 步长改为2加速
                        similarity_method=method, min_segment_frames=min_segment_frames
                    )
                    
                    # 评估分割质量
                    score = evaluate_segmentation_quality(boundaries, feats.shape[0], fps)
                    
                    if score > best_score:
                        best_boundaries = boundaries
                        best_score = score
                        best_params = {
                            'window_size': window_size,
                            'threshold': threshold,
                            'method': method,
                            'num_boundaries': len(boundaries),
                            'score': score
                        }
                
                except Exception as e:
                    continue
    
    return best_boundaries, best_params

def evaluate_segmentation_quality(boundaries, total_frames, fps=10):
    """
    评估分割质量
    """
    if len(boundaries) == 0:
        # 单段视频
        return 0.7 if total_frames >= fps * 5 else 0.5
    
    # 计算段长度
    segment_lengths = []
    prev = 0
    for boundary in boundaries + [total_frames]:
        segment_lengths.append(boundary - prev)
        prev = boundary
    
    segment_durations = [length / fps for length in segment_lengths]
    
    # 评估指标
    min_duration = min(segment_durations)
    avg_duration = np.mean(segment_durations)
    num_segments = len(segment_durations)
    std_duration = np.std(segment_durations)
    
    score = 1.0
    
    # 1. 最短段检查
    if min_duration < 1.0:
        score *= 0.2
    elif min_duration < 1.5:
        score *= 0.7
    
    # 2. 段数检查  
    if num_segments > 6:
        score *= 0.3
    elif num_segments > 4:
        score *= 0.8
    
    # 3. 平均长度检查
    if avg_duration < 1.5:
        score *= 0.4
    elif avg_duration > 6.0:
        score *= 0.7
    
    # 4. 一致性检查
    if std_duration > avg_duration * 0.8:
        score *= 0.8
    
    return score

def enhanced_sliding_window(feats, fps=10):
    """
    增强的滑动窗口算法 - 结合传统方法的优点
    """
    print(f"  Enhanced sliding window segmentation")
    
    # 预处理：轻度标准化
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)
    
    # 1. 基础滑动窗口检测
    boundaries_sw, params_sw = adaptive_sliding_window(feats_scaled, fps, debug=False)
    
    # 2. 传统散度方法作为补充
    boundaries_div = []
    try:
        # 简化的散度计算
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2_dist = np.linalg.norm(diffs, axis=1)
        
        # 平滑
        if len(l2_dist) >= 5:
            window = min(7, len(l2_dist) // 4)
            if window % 2 == 0:
                window += 1
            if window >= 3:
                l2_smooth = savgol_filter(l2_dist, window, 1, mode='mirror')
                
                # 简单阈值检测
                threshold = np.percentile(l2_smooth, 85)
                candidates = np.where(l2_smooth > threshold)[0]
                
                # 过滤相近的候选点
                for candidate in candidates:
                    if not boundaries_div or candidate - boundaries_div[-1] >= fps:
                        boundaries_div.append(candidate)
    except:
        pass
    
    # 3. 融合两种方法的结果
    all_boundaries = set(boundaries_sw + boundaries_div)
    
    # 4. 后处理：去除过近的边界点
    sorted_boundaries = sorted(all_boundaries)
    final_boundaries = []
    
    for boundary in sorted_boundaries:
        if not final_boundaries or boundary - final_boundaries[-1] >= fps:
            final_boundaries.append(boundary)
    
    # 5. 质量检查
    score = evaluate_segmentation_quality(final_boundaries, feats.shape[0], fps)
    
    # 6. 如果质量不好，选择更保守的结果
    if score < 0.5:
        if len(boundaries_sw) <= len(boundaries_div):
            final_boundaries = boundaries_sw
            method_used = 'sliding_window_only'
        else:
            final_boundaries = boundaries_div
            method_used = 'divergence_only'
    else:
        method_used = 'hybrid_sw_div'
    
    final_params = {
        **params_sw,
        'method_used': method_used,
        'sw_boundaries': len(boundaries_sw),
        'div_boundaries': len(boundaries_div),
        'final_boundaries': len(final_boundaries),
        'final_score': score
    }
    
    return final_boundaries, final_params

def visualize_sliding_window(feats, boundaries, similarities, positions, name, save_dir='sw_plots'):
    """
    可视化滑动窗口结果
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))
    
    # 上图：特征的L2范数变化
    feature_norms = np.linalg.norm(feats, axis=1)
    time_axis = np.arange(len(feature_norms)) / 10.0  # 转换为秒
    
    ax1.plot(time_axis, feature_norms, 'b-', alpha=0.7, linewidth=1)
    for boundary in boundaries:
        ax1.axvline(x=boundary/10.0, color='red', linewidth=2, alpha=0.8)
        ax1.text(boundary/10.0, ax1.get_ylim()[1]*0.9, f'{boundary/10.0:.1f}s', 
                rotation=90, ha='right', va='top')
    
    ax1.set_title(f'{name} - Feature Norms and Detected Boundaries')
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Feature L2 Norm')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 10)
    
    # 下图：相似度变化
    if similarities and positions:
        sim_time = np.array(positions) / 10.0
        ax2.plot(sim_time, similarities, 'g-', alpha=0.8, linewidth=1, label='Window Similarity')
        
        # 标记阈值线
        if len(similarities) > 0:
            # 假设使用了0.8的阈值
            threshold_line = np.full_like(similarities, 0.8)
            ax2.plot(sim_time, threshold_line, 'r--', alpha=0.7, label='Threshold')
        
        for boundary in boundaries:
            ax2.axvline(x=boundary/10.0, color='red', linewidth=2, alpha=0.8)
    
    ax2.set_title('Sliding Window Similarity')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Similarity')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{name}_sliding_window.png'), dpi=150, bbox_inches='tight')
    plt.close()

if __name__ == '__main__':
    feature_dir = '/media/wang/New Volume1/josie/GEBD-main/clip_embeddings_by_video (copy)'
    fps = 10
    all_results = {}
    
    # 统计信息
    method_stats = {}
    score_stats = []
    failed_videos = []
    
    # 设置
    PROCESS_ALL_DATA = True  # 处理全部数据
    MAX_VIDEOS = None        # 不限制数量
    CREATE_PLOTS = False     # 关闭可视化以提高速度
    SAVE_EVERY = 200         # 每200个视频保存一次
    
    print("🎯 启动滑动窗口视频分割算法 - 全数据集处理...")
    print("📊 核心思想: 相邻窗口相似度低于阈值时检测边界")
    
    # 获取文件列表
    all_files = glob.glob(os.path.join(feature_dir, '*.pt'))
    total_files = len(all_files)
    
    if MAX_VIDEOS:
        all_files = all_files[:MAX_VIDEOS]
        print(f"📁 限制处理 {len(all_files)} 个视频（总共{total_files}个）")
    else:
        print(f"📁 将处理全部 {total_files} 个视频")
    
    processed_count = 0
    for fp in tqdm(all_files, desc="Processing videos"):
        name = os.path.splitext(os.path.basename(fp))[0]
        
        try:
            # 加载数据
            data = torch.load(fp, map_location='cpu')
            
            # 解析特征
            feats = None
            if isinstance(data, torch.Tensor):
                feats = data.numpy()
            elif isinstance(data, dict):
                for key in ['features', 'clip_features', 'embeddings', 'data']:
                    if key in data:
                        arr = data[key]
                        feats = arr.numpy() if isinstance(arr, torch.Tensor) else np.array(arr)
                        break
            
            if feats is None or feats.size == 0 or len(feats.shape) != 2:
                failed_videos.append(name)
                continue
            
            # 简化输出 - 每100个视频打印一次进度
            if processed_count % 100 == 0:
                print(f"\n📹 Progress: {processed_count}/{len(all_files)} videos processed")
                print(f"   Current: {name} ({feats.shape[0]} frames)")
            
            # 执行滑动窗口分割
            boundaries, params = enhanced_sliding_window(feats, fps)
            
            # 统计
            method_used = params.get('method_used', 'unknown')
            if method_used not in method_stats:
                method_stats[method_used] = 0
            method_stats[method_used] += 1
            
            score = params.get('final_score', 0)
            score_stats.append(score)
            
            # 创建可视化（只针对前5个视频）
            if CREATE_PLOTS and processed_count < 5:
                try:
                    _, similarities, positions = sliding_window_segmentation(
                        feats, window_size=params.get('window_size', 10), 
                        threshold=params.get('threshold', 0.8),
                        similarity_method=params.get('method', 'cosine')
                    )
                    visualize_sliding_window(feats, boundaries, similarities, positions, name)
                except:
                    pass
            
            # 构建结果
            T = feats.shape[0]
            segments = []
            prev = 0
            for boundary in boundaries:
                if boundary > prev:
                    segments.append({'start': int(prev), 'end': int(boundary)})
                    prev = boundary + 1
            if prev < T:
                segments.append({'start': int(prev), 'end': int(T - 1)})

            duration = T / fps
            all_results[name] = {
                'num_frames': int(T),
                'path_video': f'{name}.mp4',
                'fps': float(fps),
                'video_duration': round(float(duration), 5),
                'substages_myframeidx': [[seg['start'], seg['end']] for seg in segments],
                'substages_timestamps': [
                    [round(seg['start']/fps,5), round(seg['end']/fps,5)]
                    for seg in segments
                ],
                'method': str(method_used),
                'params': convert_to_serializable(params),
                'sliding_window_algorithm': True
            }
            
            processed_count += 1
            
            # 定期保存，防止数据丢失
            if processed_count % SAVE_EVERY == 0:
                temp_path = f'sliding_window_temp_{processed_count}.json'
                with open(temp_path, 'w', encoding='utf-8') as f:
                    json.dump(all_results, f, ensure_ascii=False, indent=2)
                print(f"\n💾 临时保存: {processed_count}/{len(all_files)} 视频 -> {temp_path}")
                print(f"   当前平均分: {np.mean(score_stats):.3f}")
            
        except Exception as e:
            failed_videos.append(name)
            # 每50个失败打印一次
            if len(failed_videos) % 50 == 0:
                print(f"\n⚠️  累计失败: {len(failed_videos)} 个视频")
            continue

    # 最终报告
    print(f"\n📋 滑动窗口算法完整处理报告:")
    print(f"  总视频数: {total_files}")
    print(f"  成功处理: {len(all_results)} 个视频")
    print(f"  失败: {len(failed_videos)} 个视频")
    print(f"  成功率: {len(all_results)/total_files*100:.1f}%")
    
    if score_stats:
        print(f"  平均质量分: {np.mean(score_stats):.3f}")
        print(f"  分数范围: {np.min(score_stats):.2f} - {np.max(score_stats):.2f}")
        print(f"  高分(>0.8)视频: {sum(1 for s in score_stats if s > 0.8)} 个")
    
    if method_stats:
        print(f"\n📈 方法使用统计:")
        for method, count in sorted(method_stats.items()):
            print(f"  {method}: {count} 个视频 ({count/len(all_results)*100:.1f}%)")

    # 保存结果
    out_path = 'segmentation_results_sliding_window11.12extended.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ 滑动窗口算法完成! 结果保存到: {out_path}")
    print(f"📊 共处理 {len(all_results)} 个视频")
    
    # 清理临时文件
    temp_files = glob.glob('sliding_window_temp_*.json')
    if temp_files:
        print(f"\n🗑️  清理 {len(temp_files)} 个临时文件")
        for temp_file in temp_files:
            try:
                os.remove(temp_file)
                print(f"   删除: {temp_file}")
            except:
                pass
    
    if CREATE_PLOTS:
        print(f"📊 前5个视频的可视化图保存到 sw_plots/ 目录")
    
    print(f"\n🎯 滑动窗口算法特点总结:")
    print(f"  1. 核心思想: 相邻窗口不相似 → 边界点")
    print(f"  2. 主要参数: 窗口大小(1-1.5s) + 相似度阈值(0.8-0.85)")
    print(f"  3. 计算效率: 优化后的参数搜索，适合大规模处理")
    print(f"  4. 鲁棒性: 窗口平均减少单帧噪声影响")
    print(f"  5. 混合策略: 结合传统散度方法的优点")
    print(f"  6. 数据安全: 每{SAVE_EVERY}个视频自动保存，防止数据丢失")
    
    print(f"\n💡 预期改进:")
    print(f"  - 相比传统方法，应该有更稳定的边界检测")
    print(f"  - 减少噪声导致的假阳性")
    print(f"  - 更直观的参数调节")
    print(f"  - F1分数应该有所提升")

🎯 启动滑动窗口视频分割算法 - 全数据集处理...
📊 核心思想: 相邻窗口相似度低于阈值时检测边界
📁 将处理全部 1000 个视频


Processing videos:   2%|▏         | 18/1000 [00:00<00:05, 172.55it/s]


📹 Progress: 0/1000 videos processed
   Current: --SrV9bGzRA_000197_000207 (84 frames)
  Enhanced sliding window segmentation
    Sliding window: size=10, step=2, threshold=0.8
      Boundary found at frame 10, similarity=0.694
      Boundary found at frame 20, similarity=0.211
      Boundary found at frame 30, similarity=0.476
      Boundary found at frame 40, similarity=0.247
      Boundary found at frame 50, similarity=0.476
      Boundary found at frame 60, similarity=0.193
      Boundary found at frame 70, similarity=0.743
    Sliding window: size=10, step=2, threshold=0.85
      Boundary found at frame 10, similarity=0.694
      Boundary found at frame 20, similarity=0.211
      Boundary found at frame 30, similarity=0.476
      Boundary found at frame 40, similarity=0.247
      Boundary found at frame 50, similarity=0.476
      Boundary found at frame 60, similarity=0.193
      Boundary found at frame 70, similarity=0.743
    Sliding window: size=15, step=2, threshold=0.8
      

Processing videos:   6%|▌         | 56/1000 [00:00<00:05, 181.54it/s]

    Sliding window: size=10, step=2, threshold=0.8
      Boundary found at frame 10, similarity=0.243
      Boundary found at frame 20, similarity=0.104
      Boundary found at frame 30, similarity=0.244
      Boundary found at frame 40, similarity=0.345
      Boundary found at frame 50, similarity=0.418
      Boundary found at frame 60, similarity=0.214
      Boundary found at frame 70, similarity=0.227
      Boundary found at frame 80, similarity=0.249
      Boundary found at frame 90, similarity=0.291
    Sliding window: size=10, step=2, threshold=0.85
      Boundary found at frame 10, similarity=0.243
      Boundary found at frame 20, similarity=0.104
      Boundary found at frame 30, similarity=0.244
      Boundary found at frame 40, similarity=0.345
      Boundary found at frame 50, similarity=0.418
      Boundary found at frame 60, similarity=0.214
      Boundary found at frame 70, similarity=0.227
      Boundary found at frame 80, similarity=0.249
      Boundary found at frame 

Processing videos:   9%|▉         | 93/1000 [00:00<00:05, 178.50it/s]

... (49 outputs left out)

Processing videos:  98%|█████████▊| 980/1000 [00:05<00:00, 178.83it/s]

      Boundary found at frame 47, similarity=-0.477
      Boundary found at frame 63, similarity=0.365
      Boundary found at frame 79, similarity=0.507
    Sliding window: size=15, step=2, threshold=0.85
      Boundary found at frame 15, similarity=0.548
      Boundary found at frame 31, similarity=0.574
      Boundary found at frame 47, similarity=-0.477
      Boundary found at frame 63, similarity=0.365
      Boundary found at frame 79, similarity=0.507
  Enhanced sliding window segmentation
    Sliding window: size=10, step=2, threshold=0.8
      Boundary found at frame 10, similarity=0.045
      Boundary found at frame 20, similarity=0.180
      Boundary found at frame 30, similarity=0.345
      Boundary found at frame 40, similarity=0.537
      Boundary found at frame 50, similarity=0.273
      Boundary found at frame 60, similarity=-0.053
      Boundary found at frame 70, similarity=0.266
      Boundary found at frame 80, similarity=0.747
    Sliding window: size=10, step=2, th

Processing videos: 100%|██████████| 1000/1000 [00:05<00:00, 172.84it/s]

    Sliding window: size=10, step=2, threshold=0.8
      Boundary found at frame 10, similarity=0.288
      Boundary found at frame 20, similarity=0.646
      Boundary found at frame 30, similarity=0.589
      Boundary found at frame 40, similarity=-0.529
      Boundary found at frame 50, similarity=0.543
      Boundary found at frame 60, similarity=0.490
      Boundary found at frame 70, similarity=0.401
      Boundary found at frame 80, similarity=0.045
      Boundary found at frame 90, similarity=0.400
    Sliding window: size=10, step=2, threshold=0.85
      Boundary found at frame 10, similarity=0.288
      Boundary found at frame 20, similarity=0.646
      Boundary found at frame 30, similarity=0.589
      Boundary found at frame 40, similarity=-0.529
      Boundary found at frame 50, similarity=0.543
      Boundary found at frame 60, similarity=0.490
      Boundary found at frame 70, similarity=0.401
      Boundary found at frame 80, similarity=0.045
      Boundary found at fram

# extended sliding window

In [2]:
import os, glob, json
import numpy as np
import torch
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import matplotlib.pyplot as plt

def convert_to_serializable(obj):
    """JSON序列化转换"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    else:
        return obj

def compute_window_similarity(window_a, window_b, method='cosine'):
    """
    计算两个窗口的相似度
    Args:
        window_a, window_b: (window_size, feature_dim) 的特征窗口
        method: 'cosine', 'l2', 'correlation', 'histogram'
    """
    if method == 'cosine':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        norm_a = np.linalg.norm(mean_a)
        norm_b = np.linalg.norm(mean_b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        cosine_sim = np.dot(mean_a, mean_b) / (norm_a * norm_b)
        return float(cosine_sim)

    elif method == 'l2':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        l2_dist = np.linalg.norm(mean_a - mean_b)
        similarity = np.exp(-l2_dist)
        return float(similarity)

    elif method == 'correlation':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        if np.std(mean_a) == 0 or np.std(mean_b) == 0:
            return 1.0 if np.allclose(mean_a, mean_b) else 0.0
        correlation = np.corrcoef(mean_a, mean_b)[0, 1]
        return float(correlation) if not np.isnan(correlation) else 0.0

    elif method == 'histogram':
        hist_a, _ = np.histogram(window_a.flatten(), bins=50, density=True)
        hist_b, _ = np.histogram(window_b.flatten(), bins=50, density=True)
        intersection = np.minimum(hist_a, hist_b).sum()
        return float(intersection)

def sliding_window_segmentation(feats, window_size=10, step_size=2, threshold=0.8, 
                               similarity_method='cosine', min_segment_frames=10):
    """
    【已改为不对称策略的实现，保持原函数签名与返回格式不变】
    不对称滑动窗口分割：
      - 右窗口长度固定 = window_size
      - 左窗口初始长度 = window_size，若“像”则不断增大（左窗口变长，右窗口继续前移）
      - 当“不像”（相似度 < threshold）时，在左右交界处切分，并重置左窗口长度
      - 为防止左窗口无限变长，设定 max_left 上限；达到上限仍“像”则推进起点并重置
    参数对应关系（与旧版保持兼容）：
      - window_size: 右窗口长度 & 左窗口初始长度（帧）
      - step_size:   重置/推进起点的步长（帧）
      - threshold/similarity_method/min_segment_frames: 语义不变
    返回：
      boundaries:   边界帧索引列表（切在左右交界处）
      similarities: 每次评估的相似度序列
      positions:    每次相似度对应的交界帧位置（用于可视化）
    """
    T, D = feats.shape
    boundaries, similarities, positions = [], [], []

    right_size = int(window_size)
    left_init  = int(window_size)
    step_when_reset = int(step_size)

    # 左窗口每次增长的步长与最大长度上限（经验值）
    grow = max(1, right_size // 5)        # 例如右窗=10，则每次左窗+2
    max_left = right_size * 5             # 左窗口最长 ~ 5 秒（若 fps=10）

    print(f"    Asym SW: right={right_size}, left_init={left_init}, grow={grow}, "
          f"max_left={max_left}, reset_step={step_when_reset}, thr={threshold}")

    pos = 0
    left_len = left_init

    while pos + left_len + right_size <= T:
        left_start = pos
        left_end   = pos + left_len
        right_end  = left_end + right_size

        window_a = feats[left_start:left_end]
        window_b = feats[left_end:right_end]

        sim = compute_window_similarity(window_a, window_b, similarity_method)
        similarities.append(float(sim))
        positions.append(left_end)  # 交界点位置

        if sim < threshold:
            boundary_pos = left_end
            # 最小段约束
            if (not boundaries) or (boundary_pos - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(boundary_pos))
                # 从新段起点重置
                pos = boundary_pos
                left_len = left_init
                continue
            else:
                # 太近则继续累积一点再看
                left_len = min(left_len + grow, max_left)
                if (pos + left_len + right_size > T) or (left_len >= max_left):
                    pos += step_when_reset
                    left_len = left_init
                continue
        else:
            # 仍然“像”，扩大左窗口
            left_len = min(left_len + grow, max_left)
            if (pos + left_len + right_size > T) or (left_len >= max_left):
                pos += step_when_reset
                left_len = left_init

    return boundaries, similarities, positions

def adaptive_sliding_window(feats, fps=10, debug=False):
    """
    自适应滑动窗口（调用已改为不对称策略的 sliding_window_segmentation）
    搜索少量参数组合，按启发式质量分选最优
    """
    if debug:
        print(f"  Adaptive sliding window on {feats.shape[0]} frames")

    window_sizes = [10, 15]  # 1s / 1.5s
    thresholds = [0.8, 0.85]
    methods = ['cosine']

    best_boundaries = []
    best_score = 0
    best_params = {}

    for window_size in window_sizes:
        for threshold in thresholds:
            for method in methods:
                try:
                    # 最短段长度：至少 1 秒或 >= 窗口
                    min_segment_frames = max(fps, window_size)

                    # 注意：此处 step_size 作为“reset步长”传递进不对称版本
                    boundaries, similarities, positions = sliding_window_segmentation(
                        feats,
                        window_size=window_size,
                        step_size=2,                 # 重置推进步长
                        threshold=threshold,
                        similarity_method=method,
                        min_segment_frames=min_segment_frames
                    )

                    score = evaluate_segmentation_quality(boundaries, feats.shape[0], fps)

                    if score > best_score:
                        best_boundaries = boundaries
                        best_score = score
                        best_params = {
                            'window_size': window_size,
                            'threshold': threshold,
                            'method': method,
                            'num_boundaries': len(boundaries),
                            'score': score
                        }
                except Exception:
                    continue

    return best_boundaries, best_params

def evaluate_segmentation_quality(boundaries, total_frames, fps=10):
    """
    无监督启发式质量分：惩罚过短段/段数过多/平均过短或过长/时长离散过大
    """
    if len(boundaries) == 0:
        # 单段视频
        return 0.7 if total_frames >= fps * 5 else 0.5

    segment_lengths = []
    prev = 0
    for boundary in boundaries + [total_frames]:
        segment_lengths.append(boundary - prev)
        prev = boundary

    segment_durations = [length / fps for length in segment_lengths]

    min_duration = min(segment_durations)
    avg_duration = np.mean(segment_durations)
    num_segments = len(segment_durations)
    std_duration = np.std(segment_durations)

    score = 1.0
    if min_duration < 1.0:
        score *= 0.2
    elif min_duration < 1.5:
        score *= 0.7

    if num_segments > 6:
        score *= 0.3
    elif num_segments > 4:
        score *= 0.8

    if avg_duration < 1.5:
        score *= 0.4
    elif avg_duration > 6.0:
        score *= 0.7

    if std_duration > avg_duration * 0.8:
        score *= 0.8

    return score

def enhanced_sliding_window(feats, fps=10):
    """
    增强版：标准化 + 不对称SW（自适应） + 差分-L2-平滑候选 + 融合 + 质量兜底
    """
    print(f"  Enhanced sliding window segmentation")

    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)

    # 1) 不对称SW（自适应）
    boundaries_sw, params_sw = adaptive_sliding_window(feats_scaled, fps, debug=False)

    # 2) 差分-L2-平滑阈值（传统补充）
    boundaries_div = []
    try:
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2_dist = np.linalg.norm(diffs, axis=1)

        if len(l2_dist) >= 5:
            window = min(7, len(l2_dist) // 4)
            if window % 2 == 0:
                window += 1
            if window >= 3:
                l2_smooth = savgol_filter(l2_dist, window, 1, mode='mirror')
                thr = np.percentile(l2_smooth, 85)
                candidates = np.where(l2_smooth > thr)[0]

                for c in candidates:
                    if not boundaries_div or c - boundaries_div[-1] >= fps:
                        boundaries_div.append(int(c))
    except Exception:
        pass

    # 3) 融合 & 去近邻
    all_boundaries = set(boundaries_sw + boundaries_div)
    sorted_boundaries = sorted(all_boundaries)
    final_boundaries = []
    for b in sorted_boundaries:
        if not final_boundaries or b - final_boundaries[-1] >= fps:
            final_boundaries.append(int(b))

    # 4) 质量评估与兜底
    score = evaluate_segmentation_quality(final_boundaries, feats.shape[0], fps)
    if score < 0.5:
        if len(boundaries_sw) <= len(boundaries_div):
            final_boundaries = boundaries_sw
            method_used = 'divergence_only'
        else:
            final_boundaries = boundaries_sw
            method_used = 'sliding_window_only'
    else:
        method_used = 'hybrid_sw_div'

    final_params = {
        **params_sw,
        'method_used': method_used,
        'sw_boundaries': len(boundaries_sw),
        'div_boundaries': len(boundaries_div),
        'final_boundaries': len(final_boundaries),
        'final_score': score
    }

    return final_boundaries, final_params

def visualize_sliding_window(feats, boundaries, similarities, positions, name, save_dir='sw_plots'):
    """
    可视化不对称SW：上图特征范数，下图相似度曲线与阈值线
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

    feature_norms = np.linalg.norm(feats, axis=1)
    time_axis = np.arange(len(feature_norms)) / 10.0

    ax1.plot(time_axis, feature_norms, 'b-', alpha=0.7, linewidth=1)
    for boundary in boundaries:
        ax1.axvline(x=boundary/10.0, color='red', linewidth=2, alpha=0.8)
        ax1.text(boundary/10.0, ax1.get_ylim()[1]*0.9, f'{boundary/10.0:.1f}s', 
                 rotation=90, ha='right', va='top')

    ax1.set_title(f'{name} - Feature Norms and Detected Boundaries')
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Feature L2 Norm')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 10)

    if similarities and positions:
        sim_time = np.array(positions) / 10.0
        ax2.plot(sim_time, similarities, 'g-', alpha=0.8, linewidth=1, label='Window Similarity')
        if len(similarities) > 0:
            threshold_line = np.full_like(similarities, 0.8)
            ax2.plot(sim_time, threshold_line, 'r--', alpha=0.7, label='Threshold')

        for boundary in boundaries:
            ax2.axvline(x=boundary/10.0, color='red', linewidth=2, alpha=0.8)

    ax2.set_title('Sliding Window Similarity (Asymmetric)')
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Similarity')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{name}_sliding_window.png'), dpi=150, bbox_inches='tight')
    plt.close()

if __name__ == '__main__':
    feature_dir = '/media/wang/New Volume1/josie/GEBD-main/clip_embeddings_by_video (copy)'
    fps = 10
    all_results = {}

    method_stats = {}
    score_stats = []
    failed_videos = []

    PROCESS_ALL_DATA = True
    MAX_VIDEOS = None
    CREATE_PLOTS = False
    SAVE_EVERY = 200

    print("🎯 启动滑动窗口视频分割算法（不对称窗口版） - 全数据集处理...")
    print("📊 核心思想: 左窗口逐步累积上下文，右窗口固定前移；相似度低于阈值即在交界处切分")

    all_files = glob.glob(os.path.join(feature_dir, '*.pt'))
    total_files = len(all_files)

    if MAX_VIDEOS:
        all_files = all_files[:MAX_VIDEOS]
        print(f"📁 限制处理 {len(all_files)} 个视频（总共{total_files}个）")
    else:
        print(f"📁 将处理全部 {total_files} 个视频")

    processed_count = 0
    for fp in tqdm(all_files, desc="Processing videos"):
        name = os.path.splitext(os.path.basename(fp))[0]

        try:
            data = torch.load(fp, map_location='cpu')

            feats = None
            if isinstance(data, torch.Tensor):
                feats = data.numpy()
            elif isinstance(data, dict):
                for key in ['features', 'clip_features', 'embeddings', 'data']:
                    if key in data:
                        arr = data[key]
                        feats = arr.numpy() if isinstance(arr, torch.Tensor) else np.array(arr)
                        break

            if feats is None or feats.size == 0 or len(feats.shape) != 2:
                failed_videos.append(name)
                continue

            if processed_count % 100 == 0:
                print(f"\n📹 Progress: {processed_count}/{len(all_files)} videos processed")
                print(f"   Current: {name} ({feats.shape[0]} frames)")

            boundaries, params = enhanced_sliding_window(feats, fps)

            method_used = params.get('method_used', 'unknown')
            method_stats[method_used] = method_stats.get(method_used, 0) + 1

            score = params.get('final_score', 0)
            score_stats.append(score)

            # 可视化（如需）
            if CREATE_PLOTS and processed_count < 5:
                try:
                    _, similarities, positions = sliding_window_segmentation(
                        feats,
                        window_size=params.get('window_size', 10),
                        step_size=2,
                        threshold=params.get('threshold', 0.8),
                        similarity_method=params.get('method', 'cosine'),
                        min_segment_frames=max(fps, params.get('window_size', 10))
                    )
                    visualize_sliding_window(feats, boundaries, similarities, positions, name)
                except Exception:
                    pass

            T = feats.shape[0]
            segments = []
            prev = 0
            for boundary in boundaries:
                if boundary > prev:
                    segments.append({'start': int(prev), 'end': int(boundary)})
                    prev = boundary  # 不再 +1，避免丢帧
            if prev < T:
                segments.append({'start': int(prev), 'end': int(T - 1)})

            duration = T / fps
            all_results[name] = {
                'num_frames': int(T),
                'path_video': f'{name}.mp4',
                'fps': float(fps),
                'video_duration': round(float(duration), 5),
                'substages_myframeidx': [[seg['start'], seg['end']] for seg in segments],
                'substages_timestamps': [
                    [round(seg['start']/fps,5), round(seg['end']/fps,5)]
                    for seg in segments
                ],
                'method': str(method_used),
                'params': convert_to_serializable(params),
                'sliding_window_algorithm': True
            }

            processed_count += 1

            if processed_count % SAVE_EVERY == 0:
                temp_path = f'sliding_window_temp_{processed_count}.json'
                with open(temp_path, 'w', encoding='utf-8') as f:
                    json.dump(all_results, f, ensure_ascii=False, indent=2)
                print(f"\n💾 临时保存: {processed_count}/{len(all_files)} 视频 -> {temp_path}")
                print(f"   当前平均分: {np.mean(score_stats):.3f}")

        except Exception as e:
            failed_videos.append(name)
            if len(failed_videos) % 50 == 0:
                print(f"\n⚠️  累计失败: {len(failed_videos)} 个视频")
            continue

    print(f"\n📋 滑动窗口算法完整处理报告:")
    print(f"  总视频数: {total_files}")
    print(f"  成功处理: {len(all_results)} 个视频")
    print(f"  失败: {len(failed_videos)} 个视频")
    print(f"  成功率: {len(all_results)/total_files*100:.1f}%")

    if score_stats:
        print(f"  平均质量分: {np.mean(score_stats):.3f}")
        print(f"  分数范围: {np.min(score_stats):.2f} - {np.max(score_stats):.2f}")
        print(f"  高分(>0.8)视频: {sum(1 for s in score_stats if s > 0.8)} 个")

    if method_stats:
        print(f"\n📈 方法使用统计:")
        for method, count in sorted(method_stats.items()):
            print(f"  {method}: {count} 个视频 ({count/len(all_results)*100:.1f}%)")

    out_path = 'segmentation_results_sliding_window11.12.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ 滑动窗口算法完成! 结果保存到: {out_path}")
    print(f"📊 共处理 {len(all_results)} 个视频")

    temp_files = glob.glob('sliding_window_temp_*.json')
    if temp_files:
        print(f"\n🗑️  清理 {len(temp_files)} 个临时文件")
        for temp_file in temp_files:
            try:
                os.remove(temp_file)
                print(f"   删除: {temp_file}")
            except Exception:
                pass

    if CREATE_PLOTS:
        print(f"📊 前5个视频的可视化图保存到 sw_plots/ 目录")

    print(f"\n🎯 滑动窗口算法特点总结（不对称版）:")
    print(f"  1. 左窗口逐步累积上下文，右窗口固定长度")
    print(f"  2. 交界处相似度低于阈值则切分，避免碎片化")
    print(f"  3. 与差分法融合，稳健性更好")
    print(f"  4. 仍保留小网格搜索与质量兜底机制")


🎯 启动滑动窗口视频分割算法（不对称窗口版） - 全数据集处理...
📊 核心思想: 左窗口逐步累积上下文，右窗口固定前移；相似度低于阈值即在交界处切分
📁 将处理全部 1000 个视频


Processing videos:   0%|          | 0/1000 [00:00<?, ?it/s]


📹 Progress: 0/1000 videos processed
   Current: --SrV9bGzRA_000197_000207 (84 frames)
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85


Processing videos:   3%|▎         | 28/1000 [00:00<00:03, 277.81it/s]

    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.

Processing videos:   7%|▋         | 67/1000 [00:00<00:02, 339.01it/s]

... (23 outputs left out)

Processing videos:  95%|█████████▌| 952/1000 [00:02<00:00, 371.70it/s]

  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.

Processing videos: 100%|██████████| 1000/1000 [00:02<00:00, 357.33it/s]

    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.85
  Enhanced sliding window segmentation
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.8
    Asym SW: right=10, left_init=10, grow=2, max_left=50, reset_step=2, thr=0.85
    Asym SW: right=15, left_init=15, grow=3, max_left=75, reset_step=2, thr=0.8
    Asym SW: right=15, left_init=15, 

In [11]:
# Cell 1 (保持不变：导入)
from glob import glob
import os.path as osp
import os
import pickle
import json
import numpy as np


In [12]:
# Cell 2 (保持不变：路径与配置)
downsample = 1
min_change_duration = 0.3

with open('/media/wang/New Volume1/josie/GEBD-main/data/export/k400_mr345_val_min_change_duration'+str(min_change_duration)+'.pkl', 'rb') as f:
    gt_dict = pickle.load(f)

exp_path = '/media/wang/New Volume1/josie/GEBD-main/data/exp_k400/'
output_seg_dir = 'detect_seg'
OUTPUT_BDY_PATH = exp_path + output_seg_dir + '/{}.pkl'


In [13]:
# === 升级版第三个 Cell：在三段式里也能对比多方法 ===

list_rec = []
list_prec = []
list_f1 = []

# 你想额外对比的 JSON（和一段式保持一致）
PRED_JSONS = [
    "segmentation_results_sliding_window_sym1.json",
    "segmentation_results_sliding_window_asym1.json",
]
CONSIST_THRESH = 0.3
DS_LIST = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

def one_threshold_match(det, gt_b, d_ratio, video_len):
    if len(det) == 0 and len(gt_b) == 0:
        return (1.0, 1.0, 1.0, 0, 0, 0)
    if len(det) == 0:
        return (0.0, 0.0, 0.0, 0, 0, len(gt_b))
    if len(gt_b) == 0:
        return (0.0, 0.0, 0.0, 0, len(det), 0)
    thr = d_ratio * video_len
    det = list(map(int, det)); gt_b = list(map(int, gt_b))
    pairs = []
    for i, p in enumerate(det):
        for j, g in enumerate(gt_b):
            d = abs(p - g)
            if d <= thr:
                pairs.append((d, i, j))
    pairs.sort(key=lambda x: x[0])
    used_d, used_g = set(), set()
    tp = 0
    for _, i, j in pairs:
        if i not in used_d and j not in used_g:
            used_d.add(i); used_g.add(j); tp += 1
    fp = len(det) - tp
    fn = len(gt_b) - tp
    rec = tp / len(gt_b) if len(gt_b) > 0 else 0.0
    prec = tp / len(det) if len(det) > 0 else 0.0
    f1 = 2 * rec * prec / (rec + prec) if (rec + prec) > 0 else 0.0
    return (rec, prec, f1, tp, fp, fn)

def eval_detect_seg_official_like(gt_dict):
    list_rec, list_prec, list_f1 = [], [], []
    for d in DS_LIST:
        tp_all = 0; num_pos_all = 0; num_det_all = 0
        for vid_id in list(gt_dict.keys()):
            if gt_dict[vid_id].get('f1_consis_avg', 1.0) < CONSIST_THRESH:
                continue
            output_bdy_path = OUTPUT_BDY_PATH.format(vid_id)
            if not os.path.exists(output_bdy_path):
                continue
            with open(output_bdy_path, 'rb') as f:
                bdy_idx_save = pickle.load(f, encoding='latin1')
            det = (np.array(bdy_idx_save['bdy_idx_list_smt']) * downsample).astype(int).tolist()
            ins_start = 0
            ins_end = gt_dict[vid_id]['num_frames'] - 1
            video_len = ins_end - ins_start + 1
            det = [x for x in det if ins_start <= x <= ins_end]
            num_det_all += len(det)
            if len(det) == 0:
                if gt_dict[vid_id]['substages_myframeidx']:
                    num_pos_all += len(gt_dict[vid_id]['substages_myframeidx'][0])
                continue
            best_tp = -1; best_num_pos = 0
            for ann in gt_dict[vid_id]['substages_myframeidx']:
                gt_b = list(ann)
                _, _, _, tp, _, _ = one_threshold_match(det, gt_b, d, video_len)
                if tp > best_tp:
                    best_tp = tp
                    best_num_pos = len(gt_b)
            if best_tp < 0:
                continue
            tp_all += best_tp
            num_pos_all += best_num_pos
        fn_all = num_pos_all - tp_all
        fp_all = num_det_all - tp_all
        rec = tp_all/(tp_all+fn_all) if num_pos_all > 0 else 1.0
        prec = tp_all/(tp_all+fp_all) if (tp_all+fp_all) > 0 else 0.0
        f1 = 2*rec*prec/(rec+prec) if (rec+prec) > 0 else 0.0
        list_rec.append(rec); list_prec.append(prec); list_f1.append(f1)
    return list_rec, list_prec, list_f1

def try_match_key(pred_keys, gt_key):
    if gt_key in pred_keys:
        return gt_key
    for pk in pred_keys:
        if pk in gt_key or gt_key in pk:
            return pk
    return None

def eval_json_official_like(gt_dict, json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        pred = json.load(f)
    pred_keys = set(pred.keys())
    list_rec, list_prec, list_f1 = [], [], []
    for d in DS_LIST:
        tp_all = 0; num_pos_all = 0; num_det_all = 0
        for vid_id, item in gt_dict.items():
            if item.get('f1_consis_avg', 1.0) < CONSIST_THRESH:
                continue
            key = try_match_key(pred_keys, vid_id)
            if key is None:
                continue
            pred_item = pred[key]
            segs = pred_item.get("substages_myframeidx", [])
            if len(segs) > 1:
                pred_b_10 = [s[0] for s in segs[1:]]
            else:
                pred_b_10 = []
            pred_fps = float(pred_item.get("fps", 10.0))
            gt_fps = float(item.get("fps", 30.0))
            video_len = int(item["num_frames"])
            scale = gt_fps / pred_fps if pred_fps > 0 else 1.0
            det = [int(round(b * scale)) for b in pred_b_10]
            det = [x for x in det if 0 <= x <= video_len - 1]
            num_det_all += len(det)
            if len(det) == 0:
                if item.get('substages_myframeidx'):
                    num_pos_all += len(item['substages_myframeidx'][0])
                continue
            best_tp = -1; best_num_pos = 0
            for ann in item.get('substages_myframeidx', []):
                gt_b = list(ann)
                _, _, _, tp, _, _ = one_threshold_match(det, gt_b, d, video_len)
                if tp > best_tp:
                    best_tp = tp
                    best_num_pos = len(gt_b)
            if best_tp < 0:
                continue
            tp_all += best_tp
            num_pos_all += best_num_pos
        fn_all = num_pos_all - tp_all
        fp_all = num_det_all - tp_all
        rec = tp_all/(tp_all+fn_all) if num_pos_all > 0 else 1.0
        prec = tp_all/(tp_all+fp_all) if (tp_all+fp_all) > 0 else 0.0
        f1 = 2*rec*prec/(rec+prec) if (rec+prec) > 0 else 0.0
        list_rec.append(rec); list_prec.append(prec); list_f1.append(f1)
    return list_rec, list_prec, list_f1

# 1) 官方 detect_seg（pkl）——复用你原路径
rec_d, prec_d, f1_d = eval_detect_seg_official_like(gt_dict)
print("=== GEBD detect_seg (PA–DPC) ===")
print("rec: " + str(np.mean(rec_d)))
print("prec: " + str(np.mean(prec_d)))
print("F1: " + str(np.mean(f1_d)))
print("rec: " + str(rec_d))
print("prec: " + str(prec_d))
print("F1: " + str(f1_d))

# 2) 你的 JSON 方法逐个评测并对比
results_summary = [("GEBD detect_seg (PA–DPC)", np.mean(rec_d), np.mean(prec_d), np.mean(f1_d))]
for jp in PRED_JSONS:
    if not os.path.isfile(jp):
        print(f"[跳过] 找不到 {jp}")
        continue
    rec_j, prec_j, f1_j = eval_json_official_like(gt_dict, jp)
    print(f"\n=== {os.path.basename(jp)} (official protocol) ===")
    print("rec: " + str(np.mean(rec_j)))
    print("prec: " + str(np.mean(prec_j)))
    print("F1: " + str(np.mean(f1_j)))
    print("rec: " + str(rec_j))
    print("prec: " + str(prec_j))
    print("F1: " + str(f1_j))
    results_summary.append((os.path.basename(jp), np.mean(rec_j), np.mean(prec_j), np.mean(f1_j)))

# 3) 打一个简单的对比表（按 mean_F1 排序）
if results_summary:
    import pandas as pd
    df = pd.DataFrame(results_summary, columns=["method","mean_rec","mean_prec","mean_F1"]).sort_values("mean_F1", ascending=False)
    print("\n=== 对比排行（GEBD 官方协议：mean F1 over d∈[0.05,0.5]） ===")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# 4) 保存“官方 detect_seg”的三条曲线（保持你原有保存逻辑）
np.save(exp_path + output_seg_dir + '.eval.mindur'+str(min_change_duration)+'.npy', [rec_d, prec_d, f1_d])


=== GEBD detect_seg (PA–DPC) ===
rec: 0.24846598798167907
prec: 0.9276641091219098
F1: 0.39191654418103894
rec: [0.1852033852033852, 0.22583658442771581, 0.2415011780545271, 0.2487410173711311, 0.2539320720310007, 0.2590679522497704, 0.2631578947368421, 0.26622431756009546, 0.2692352666043031, 0.2717602115780194]
prec: [0.7229326513213982, 0.8759590792838875, 0.9175191815856778, 0.9369138959931799, 0.9497016197783461, 0.9620630861040068, 0.9697357203751066, 0.974850809889173, 0.981457800511509, 0.9855072463768116]
F1: [0.2948667796757508, 0.35909309335546724, 0.3823607780442313, 0.3931142410015649, 0.4007194244604317, 0.4082112497739193, 0.41397507051223725, 0.4182325241164907, 0.42255459717379334, 0.4260376836965034]

=== segmentation_results_sliding_window_sym1.json (official protocol) ===
rec: 0.7024044292029407
prec: 0.8761454888993858
F1: 0.7797026009060037
rec: [0.5483564506935208, 0.6649214659685864, 0.687229640774873, 0.6960692119616325, 0.7144469525959368, 0.7273586689355266, 

# compare two window asym/sym

In [9]:
import os, glob, json
import numpy as np
import torch
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import matplotlib.pyplot as plt

# ===================== 公共工具 =====================
def convert_to_serializable(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: convert_to_serializable(v) for k,v in obj.items()}
    if isinstance(obj, list): return [convert_to_serializable(x) for x in obj]
    return obj

def compute_window_similarity(window_a, window_b, method='cosine'):
    if method == 'cosine':
        mean_a = np.mean(window_a, axis=0); mean_b = np.mean(window_b, axis=0)
        na, nb = np.linalg.norm(mean_a), np.linalg.norm(mean_b)
        if na == 0 or nb == 0: return 0.0
        return float(np.dot(mean_a, mean_b)/(na*nb))
    elif method == 'l2':
        mean_a = np.mean(window_a, axis=0); mean_b = np.mean(window_b, axis=0)
        return float(np.exp(-np.linalg.norm(mean_a-mean_b)))
    elif method == 'correlation':
        mean_a = np.mean(window_a, axis=0); mean_b = np.mean(window_b, axis=0)
        if np.std(mean_a)==0 or np.std(mean_b)==0:
            return 1.0 if np.allclose(mean_a, mean_b) else 0.0
        r = np.corrcoef(mean_a, mean_b)[0,1]
        return float(r) if not np.isnan(r) else 0.0
    elif method == 'histogram':
        ha,_ = np.histogram(window_a.flatten(), bins=50, density=True)
        hb,_ = np.histogram(window_b.flatten(), bins=50, density=True)
        return float(np.minimum(ha,hb).sum())

# ===================== 两种窗口策略 =====================
def sw_symmetric(feats, window_size=10, step_stride=1, threshold=0.8,
                 similarity_method='cosine', min_segment_frames=10):
    """
    对称SW：左=右=window_size，步长 step_stride（此处=1 更密）。
    """
    T = feats.shape[0]
    boundaries, similarities, positions = [], [], []
    pos = 0
    while pos + 2*window_size <= T:
        a = feats[pos:pos+window_size]
        b = feats[pos+window_size:pos+2*window_size]
        sim = compute_window_similarity(a,b,similarity_method)
        similarities.append(float(sim))
        mid = pos + window_size
        positions.append(mid)
        if sim < threshold:
            if (not boundaries) or (mid - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(mid))
        pos += step_stride
    return boundaries, similarities, positions

def sw_asymmetric(feats, window_size=10, step_reset=1, threshold=0.8,
                  similarity_method='cosine', min_segment_frames=10):
    """
    不对称SW：右窗固定=window_size；左窗从window_size起，每次 +grow（=1），
    达上限(3*window)或触发切分则重置；reset 步长=1。
    """
    T = feats.shape[0]
    right_size = int(window_size)
    left_init  = int(window_size)
    grow = 1
    max_left = right_size * 3

    boundaries, similarities, positions = [], [], []
    pos, left_len = 0, left_init
    while pos + left_len + right_size <= T:
        left_start = pos; left_end = pos + left_len; right_end = left_end + right_size
        a = feats[left_start:left_end]; b = feats[left_end:right_end]
        sim = compute_window_similarity(a,b,similarity_method)
        similarities.append(float(sim)); positions.append(left_end)
        if sim < threshold:
            mid = left_end
            if (not boundaries) or (mid - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(mid))
                pos = mid; left_len = left_init
            else:
                left_len = min(left_len + grow, max_left)
                if (pos + left_len + right_size > T) or (left_len >= max_left):
                    pos += step_reset; left_len = left_init
        else:
            left_len = min(left_len + grow, max_left)
            if (pos + left_len + right_size > T) or (left_len >= max_left):
                pos += step_reset; left_len = left_init
    return boundaries, similarities, positions

# ===================== 质量分 / 精修 / 融合 =====================
def evaluate_segmentation_quality(boundaries, total_frames, fps=10):
    if len(boundaries)==0:
        return 0.7 if total_frames >= fps*5 else 0.5
    seg_lens, prev = [], 0
    for b in boundaries + [total_frames]:
        seg_lens.append(b-prev); prev=b
    durs = [l/fps for l in seg_lens]
    mn, av, n, sd = min(durs), np.mean(durs), len(durs), np.std(durs)
    score = 1.0
    if mn < 1.0: score *= 0.2
    elif mn < 1.5: score *= 0.7
    if n > 6: score *= 0.3
    elif n > 4: score *= 0.8
    if av < 1.5: score *= 0.4
    elif av > 6.0: score *= 0.7
    if sd > av*0.8: score *= 0.8
    return score

def refine_boundary(b, feats, fps, radius=None):
    """
    精修：±0.3s 内选帧间差分 L2 平滑曲线的局部最大作为边界。
    半径缩小到 0.3s，避免右移过头导致召回掉线。
    """
    if radius is None:
        radius = max(2, int(0.3 * fps))   # 0.3s
    L = feats.shape[0]
    start = max(2, b - radius); end = min(L - 2, b + radius)
    diffs = feats[1:] - feats[:-1]
    l2 = np.linalg.norm(diffs, axis=1)
    l2_smooth = np.convolve(l2, np.ones(3)/3, mode='same') if len(l2)>=3 else l2
    idx = np.argmax(l2_smooth[start:end+1])
    return int(start+idx)

def post_fuse_and_refine(boundaries_a, boundaries_b, feats_scaled, fps):
    """
    融合两路候选，采用 0.3s 的最小间隔去重，再做精修（同样 0.3s 的最小间隔）。
    """
    MIN_DIST = max(1, int(0.3 * fps))   # 0.3s
    merged = sorted(set(boundaries_a + boundaries_b))
    final = []
    for b in merged:
        if not final or b - final[-1] >= MIN_DIST:
            final.append(int(b))
    # 精修
    refined = []
    for b in final:
        rb = refine_boundary(b, feats_scaled, fps)
        if not refined or rb - refined[-1] >= MIN_DIST:
            refined.append(rb)
    return refined

# ===================== 自适应搜索（统一网格） =====================
def adaptive_on_policy(feats_scaled, fps, policy='symmetric', debug=False):
    # 更密网格：窗口 10/12/15；阈值 0.72/0.74/0.76/0.80
    window_sizes = [10, 12, 15]
    thresholds   = [0.72, 0.74, 0.76, 0.80]
    methods      = ['cosine']

    best_boundaries, best_score, best_params = [], 0.0, {}
    for ws in window_sizes:
        for thr in thresholds:
            for m in methods:
                try:
                    min_seg = max(fps, ws)
                    if policy == 'symmetric':
                        boundaries, sims, pos = sw_symmetric(
                            feats_scaled, window_size=ws, step_stride=1,   # 步长 1
                            threshold=thr, similarity_method=m, min_segment_frames=min_seg
                        )
                    else:
                        boundaries, sims, pos = sw_asymmetric(
                            feats_scaled, window_size=ws, step_reset=1,   # reset 1
                            threshold=thr, similarity_method=m, min_segment_frames=min_seg
                        )
                    score = evaluate_segmentation_quality(boundaries, feats_scaled.shape[0], fps)
                    if score > best_score:
                        best_boundaries, best_score = boundaries, score
                        best_params = {'window_size': ws, 'threshold': thr, 'method': m,
                                       'num_boundaries': len(boundaries), 'score': score}
                except Exception:
                    continue
    return best_boundaries, best_params

def enhanced_runner(feats, fps, policy='symmetric'):
    """
    标准化 + 指定策略自适应 + 传统差分候选 + 融合(0.3s) + 精修(±0.3s) + 质量兜底
    """
    scaler = StandardScaler(); feats_scaled = scaler.fit_transform(feats)

    # 1) 指定策略
    sw_boundaries, sw_params = adaptive_on_policy(feats_scaled, fps, policy=policy)

    # 2) 传统差分峰值（85分位），预去重间隔 0.3s
    div_boundaries = []
    try:
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2 = np.linalg.norm(diffs, axis=1)
        if len(l2) >= 5:
            win = min(7, len(l2)//4);  win += 1 if win%2==0 else 0
            if win >= 3:
                l2_s = savgol_filter(l2, win, 1, mode='mirror')
                thr = np.percentile(l2_s, 85)
                cand = np.where(l2_s > thr)[0]
                gap = max(1, int(0.3 * fps))  # 0.3s
                for c in cand:
                    if not div_boundaries or c - div_boundaries[-1] >= gap:
                        div_boundaries.append(int(c))
    except Exception:
        pass

    # 3) 融合 + 精修
    final = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled, fps)

    # 4) 质量兜底
    score = evaluate_segmentation_quality(final, feats.shape[0], fps)
    if score < 0.5:
        if len(sw_boundaries) <= len(div_boundaries):
            final = sw_boundaries; method_used = f'{policy}_only'
        else:
            final = div_boundaries; method_used = 'divergence_only'
    else:
        method_used = f'{policy}_hybrid_div'

    params = {**sw_params, 'method_used': method_used,
              'sw_boundaries': len(sw_boundaries), 'div_boundaries': len(div_boundaries),
              'final_boundaries': len(final), 'final_score': evaluate_segmentation_quality(final, feats.shape[0], fps)}
    return final, params

# ===================== 主流程：同时跑两个版本 =====================
if __name__ == '__main__':
    # ！！！按你的实际路径修改！！！
    feature_dir = '/media/wang/New Volume1/josie/GEBD-main/clip_embeddings_by_video (copy)'
    fps = 10
    SAVE_EVERY = 200
    CREATE_PLOTS = False

    print("🎯 同参数对比：对称SW vs 不对称SW（更灵敏设置）")
    print(f"[DEBUG] feature_dir = {feature_dir}")
    print(f"[DEBUG] exists? {os.path.isdir(feature_dir)}")

    # 多后缀 + 递归寻找特征文件
    patterns = [
        os.path.join(feature_dir, '*.pt'),
        os.path.join(feature_dir, '*.npy'),
        os.path.join(feature_dir, '*.pkl'),
        os.path.join(feature_dir, '**/*.pt'),
        os.path.join(feature_dir, '**/*.npy'),
        os.path.join(feature_dir, '**/*.pkl'),
    ]
    all_files = []
    for pat in patterns:
        all_files.extend(glob.glob(pat, recursive=True))
    all_files = sorted(list(set(all_files)))
    total_files = len(all_files)
    print(f"[INFO] total feature files found: {total_files}")
    if total_files == 0:
        if "New Volume" in feature_dir and "New Volume1" not in feature_dir:
            print("⚠️ Hint: 你之前也用过 '/media/wang/New Volume1/...'\n   现在是 '/media/wang/New Volume/...'? 请确认盘符是否带 1。")
        print("⚠️ 未找到特征文件，退出。")
        print("\n📋 处理报告（两版本）:")
        print("  总视频数: 0\n  成功处理: 0\n  失败: 0\n  成功率: 0.0%")
        import sys; sys.exit(0)

    results_sym, results_asym = {}, {}
    failed_videos = []
    score_sym, score_asym = [], []

    processed_count = 0
    for fp in tqdm(all_files, desc="Processing videos"):
        name = os.path.splitext(os.path.basename(fp))[0]
        try:
            # ---- 加载特征 ----
            ext = os.path.splitext(fp)[1].lower()
            feats = None
            if ext == '.npy':
                feats = np.load(fp)
            elif ext == '.pkl':
                import pickle
                with open(fp,'rb') as f: data = pickle.load(f)
                if isinstance(data, np.ndarray): feats = data
                elif isinstance(data, torch.Tensor): feats = data.numpy()
                elif isinstance(data, dict):
                    for k in ['features','clip_features','embeddings','data']:
                        if k in data:
                            arr = data[k]; feats = arr.numpy() if isinstance(arr, torch.Tensor) else np.array(arr)
                            break
            else:
                data = torch.load(fp, map_location='cpu')
                if isinstance(data, torch.Tensor): feats = data.numpy()
                elif isinstance(data, dict):
                    for k in ['features','clip_features','embeddings','data']:
                        if k in data:
                            arr = data[k]; feats = arr.numpy() if isinstance(arr, torch.Tensor) else np.array(arr)
                            break

            if feats is None or feats.size==0 or len(feats.shape)!=2:
                failed_videos.append(name); continue

            if processed_count % 100 == 0:
                print(f"\n📹 Progress: {processed_count}/{total_files} | Now: {name} ({feats.shape[0]} frames)")

            # ---- 1) 对称 ----
            b_sym, p_sym = enhanced_runner(feats, fps, policy='symmetric')
            T = feats.shape[0]
            segs = []; prev = 0
            for b in b_sym:
                if b > prev: segs.append({'start':int(prev), 'end':int(b)}); prev = b
            if prev < T: segs.append({'start':int(prev), 'end':int(T-1)})
            results_sym[name] = {
                'num_frames': int(T), 'path_video': f'{name}.mp4', 'fps': float(fps),
                'video_duration': round(T/fps,5),
                'substages_myframeidx': [[s['start'], s['end']] for s in segs],
                'substages_timestamps': [[round(s['start']/fps,5), round(s['end']/fps,5)] for s in segs],
                'method': str(p_sym.get('method_used','sym')), 'params': convert_to_serializable(p_sym),
                'sliding_window_algorithm': True
            }
            score_sym.append(p_sym.get('final_score',0))

            # ---- 2) 不对称 ----
            b_asym, p_asym = enhanced_runner(feats, fps, policy='asymmetric')
            T = feats.shape[0]
            segs = []; prev = 0
            for b in b_asym:
                if b > prev: segs.append({'start':int(prev), 'end':int(b)}); prev = b
            if prev < T: segs.append({'start':int(prev), 'end':int(T-1)})
            results_asym[name] = {
                'num_frames': int(T), 'path_video': f'{name}.mp4', 'fps': float(fps),
                'video_duration': round(T/fps,5),
                'substages_myframeidx': [[s['start'], s['end']] for s in segs],
                'substages_timestamps': [[round(s['start']/fps,5), round(s['end']/fps,5)] for s in segs],
                'method': str(p_asym.get('method_used','asym')), 'params': convert_to_serializable(p_asym),
                'sliding_window_algorithm': True
            }
            score_asym.append(p_asym.get('final_score',0))

            processed_count += 1
            if processed_count % SAVE_EVERY == 0:
                with open('segmentation_results_sliding_window_sym_temp.json','w',encoding='utf-8') as f:
                    json.dump(results_sym, f, ensure_ascii=False, indent=2)
                with open('segmentation_results_sliding_window_asym_temp.json','w',encoding='utf-8') as f:
                    json.dump(results_asym, f, ensure_ascii=False, indent=2)
                print(f"\n💾 临时保存 {processed_count}/{total_files}")

        except Exception as e:
            failed_videos.append(name); continue

    # ---- 报告 ----
    print("\n📋 处理报告：")
    print(f"  总视频数: {total_files}")
    print(f"  成功处理: {len(results_sym)}（对称） / {len(results_asym)}（不对称）")
    print(f"  失败: {len(failed_videos)}")
    success_rate = (len(results_sym)/total_files*100) if total_files>0 else 0.0
    print(f"  成功率(对称计): {success_rate:.1f}%")
    if score_sym:
        print(f"  对称 平均质量分: {np.mean(score_sym):.3f}  范围: {np.min(score_sym):.2f}-{np.max(score_sym):.2f}")
    if score_asym:
        print(f"  不对称 平均质量分: {np.mean(score_asym):.3f}  范围: {np.min(score_asym):.2f}-{np.max(score_asym):.2f}")

    # ---- 保存 ----
    out_sym = 'segmentation_results_sliding_window_sym1.json'
    out_asym = 'segmentation_results_sliding_window_asym1.json'
    with open(out_sym,'w',encoding='utf-8') as f:
        json.dump(results_sym, f, ensure_ascii=False, indent=2)
    with open(out_asym,'w',encoding='utf-8') as f:
        json.dump(results_asym, f, ensure_ascii=False, indent=2)
    print(f"\n✅ 已保存：{out_sym} / {out_asym}")


🎯 同参数对比：对称SW vs 不对称SW（更灵敏设置）
[DEBUG] feature_dir = /media/wang/New Volume1/josie/GEBD-main/clip_embeddings_by_video (copy)
[DEBUG] exists? True
[INFO] total feature files found: 1000


Processing videos:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing videos:   0%|          | 3/1000 [00:00<00:35, 28.27it/s]


📹 Progress: 0/1000 | Now: --SrV9bGzRA_000197_000207 (84 frames)


Processing videos:  10%|█         | 105/1000 [00:03<00:28, 31.80it/s]


📹 Progress: 100/1000 | Now: -Aw1GOyNtlI_000013_000023 (50 frames)


... (14 outputs left out)

Processing videos:  91%|█████████ | 906/1000 [00:30<00:02, 32.59it/s]


📹 Progress: 900/1000 | Now: 4eEgCU1Mb68_000005_000015 (65 frames)


Processing videos: 100%|██████████| 1000/1000 [00:33<00:00, 30.23it/s]



💾 临时保存 1000/1000

📋 处理报告：
  总视频数: 1000
  成功处理: 1000（对称） / 1000（不对称）
  失败: 0
  成功率(对称计): 100.0%
  对称 平均质量分: 0.815  范围: 0.50-1.00
  不对称 平均质量分: 0.815  范围: 0.50-1.00

✅ 已保存：segmentation_results_sliding_window_sym1.json / segmentation_results_sliding_window_asym1.json


# baseline

In [ ]:
# infer_pred_error_fixed.py
import torch
from torch.utils import data
from torchvision import transforms
import os
import sys
import time
import pickle
import glob
import csv
import pandas as pd
import numpy as np
from PIL import Image  # ← 修复：PIL 导入
import cv2
sys.path.append('../utils')
from augmentation import *
from tqdm import tqdm

# ------------------------- IO helpers -------------------------
def pil_loader(path):
    with open(path, 'rb') as f:
        with Image.open(f) as img:
            return img.convert('RGB')

def pil_loader_flow(path_x, path_y):
    with open(path_x, 'rb') as f:
        with Image.open(f) as img:
            x = np.asarray(img)
            x = np.round((x-127.5)*127.5/20+127.5)
            x = (np.minimum(np.maximum(x, 0.0), 255.0))
    with open(path_y, 'rb') as f:
        with Image.open(f) as img:
            y = np.asarray(img)
            y = np.round((y-127.5)*127.5/20+127.5)
            y = (np.minimum(np.maximum(y, 0.0), 255.0))
    img = 128*np.ones((x.shape[0], x.shape[1],3), dtype=np.uint8)
    img[:, :, 0] = x
    img[:, :, 1] = y
    return Image.fromarray(img).convert('RGB')

# ------------------------- Datasets -------------------------
class TAPOS_instances_3d(data.Dataset):
    def __init__(self, mode='val', transform=None, seq_len=10, num_seq=5, downsample=1,
                 epsilon=5, unit_test=False, modality='rgb', pkl_folder_name='',
                 big=False, return_label=False, pred_task='', pred_step=3):
        self.mode = mode
        self.transform = transform
        self.seq_len = seq_len
        self.num_seq = num_seq
        self.downsample = downsample
        self.epsilon = epsilon
        self.unit_test = unit_test
        self.return_label = return_label
        self.pred_step = pred_step
        self.pred_task = pred_task
        self.modality = modality
        self.pkl_folder_name = pkl_folder_name

        print('Using TAPOS (224x224)' if big else 'Using TAPOS (112x112) ')

        if big:
            if mode in ['val', 'test']:
                split = '../../data/exp_TAPOS/val_set.csv'
                video_info = pd.read_csv(split, header=None)
            else:
                raise ValueError('train mode NOT IMPLEMENTED')
        else:
            raise ValueError('NOT IMPLEMENTED')

        drop_idx = []
        print('filter out too short videos ...')
        for idx, row in tqdm(video_info.iterrows(), total=len(video_info)):
            vpath, vlen = row
            if vlen - self.num_seq*self.seq_len*self.downsample <= 10:
                drop_idx.append(idx)
        self.video_info = video_info.drop(drop_idx, axis=0)
        print('#videos kept: ' + str(len(self.video_info)))

        if mode == 'val':
            self.video_info = self.video_info.sample(frac=1, random_state=666)
        if self.unit_test:
            self.video_info = self.video_info.sample(32, random_state=666)

        print('construct windows of stride 1 from each video ...')
        self.window_info = pd.DataFrame(columns=['vpath', 'vlen', 'window_idx',
                                                 'all_window_seq_idx_block',
                                                 'current_frame_idx'])

        pkl_dir = self.pkl_folder_name
        os.makedirs(pkl_dir, exist_ok=True)

        if self.mode == 'train':
            raise ValueError('NOT IMPLEMENTED')
        else:
            # 写入统一为 rgb；读取时兼容 rbg
            pkl_name_rgb = os.path.join(pkl_dir, "window_lists.rgb.pkl")
            pkl_name_rbg = os.path.join(pkl_dir, "window_lists.rbg.pkl")  # 兼容历史
            pkl_name = pkl_name_rgb if self.modality == 'rgb' else os.path.join(pkl_dir, "window_lists.flow.pkl")

        if os.path.exists(pkl_name):
            print("skip constructing windows... use pre-computed one...")
            with open(pkl_name, "rb") as f:
                self.window_info = pickle.load(f)
        elif self.modality == 'rgb' and os.path.exists(pkl_name_rbg):
            print("found legacy rbg pickle, loading for compatibility...")
            with open(pkl_name_rbg, "rb") as f:
                self.window_info = pickle.load(f)
        else:
            for _, (vpath, vlen) in tqdm(self.video_info.iterrows(), total=len(self.video_info)):
                if vlen - self.num_seq*self.seq_len*self.downsample <= 0:
                    print("vlen:", vlen)
                    print("need:", self.num_seq*self.seq_len*self.downsample)
                    continue
                n_window = int(vlen/self.downsample) - (1+1)*self.seq_len + 1
                all_window_seq_idx_block = np.zeros((n_window, self.num_seq, self.seq_len))
                for window_idx in range(n_window):
                    start_zeropad = self.num_seq - 1 - self.pred_step
                    window_start_frame_idx = window_idx*self.downsample - start_zeropad*self.seq_len*self.downsample
                    seq_idx = np.expand_dims(np.arange(self.num_seq), -1)*self.downsample*self.seq_len + window_start_frame_idx
                    tmp = seq_idx + np.expand_dims(np.arange(self.seq_len),0)*self.downsample
                    tmp[tmp<0] = 0
                    tmp[tmp>vlen-1] = vlen-1
                    all_window_seq_idx_block[window_idx] = tmp
                    self.window_info.loc[len(self.window_info)] = [
                        vpath, vlen, window_idx,
                        all_window_seq_idx_block[window_idx],
                        all_window_seq_idx_block[window_idx][self.num_seq-self.pred_step-1][0]
                    ]
            print("len(self.window_info):", len(self.window_info))

            # 写入统一为 rgb
            with open(pkl_name_rgb if self.modality=='rgb' else pkl_name, "wb") as f:
                pickle.dump(self.window_info, f)

    def __getitem__(self, index):
        vpath, vlen, window_idx, idx_block, current_frame_idx = self.window_info.iloc[index]
        if idx_block is None:
            print(vpath)
        assert idx_block.shape == (self.num_seq, self.seq_len)
        idx_block = idx_block.reshape(self.num_seq*self.seq_len)
        if self.modality == 'flow':
            seq = [pil_loader_flow(os.path.join(vpath, 'flow_x_%05d.jpg' % (i+1)),
                                   os.path.join(vpath, 'flow_y_%05d.jpg' % (i+1))) for i in idx_block]
        else:
            seq = [pil_loader(os.path.join(vpath, 'image_%05d.jpg' % (i+1))) for i in idx_block]
        t_seq = self.transform(seq)
        (C, H, W) = t_seq[0].size()
        t_seq = torch.stack(t_seq, 0)
        t_seq = t_seq.view(self.num_seq, self.seq_len, C, H, W).transpose(1,2)
        videoid = vpath.split('/')[-2] + '_' + vpath.split('/')[-1]
        return t_seq, videoid, vlen, window_idx, current_frame_idx

    def __len__(self):
        return len(self.window_info)

class Kinetics400_full_3d(data.Dataset):
    def __init__(self, mode='val', transform=None, seq_len=10, num_seq=5, downsample=3,
                 epsilon=5, unit_test=False, big=False, return_label=False, modality='rgb',
                 pkl_folder_name='', pred_task='', pred_step=3):
        self.mode = mode
        self.transform = transform
        self.seq_len = seq_len
        self.num_seq = num_seq
        self.downsample = downsample
        self.epsilon = epsilon
        self.unit_test = unit_test
        self.return_label = return_label
        self.pred_step = pred_step
        self.modality = modality
        self.pkl_folder_name = pkl_folder_name
        self.pred_task = pred_task

        print('Using Kinetics400 GEBD data (224x224)' if big else 'Using Kinetics400 GEBD data (112x112)')

        self.action_dict_encode, self.action_dict_decode = {}, {}
        action_file = os.path.join('../../data/exp_k400/', 'classInd.txt')
        action_df = pd.read_csv(action_file, sep=',', header=None)
        for _, row in action_df.iterrows():
            act_id, act_name = row
            act_id = int(act_id) - 1
            self.action_dict_decode[act_id] = act_name
            self.action_dict_encode[act_name] = act_id

        if big:
            if mode in ['val', 'test']:
                split = '../../data/exp_k400/val_set.csv'
                video_info = pd.read_csv(split, header=None)
            else:
                raise ValueError('train mode NOT IMPLEMENTED')
        else:
            raise ValueError('NOT IMPLEMENTED')

        drop_idx = []
        print('filter out too short videos ...')
        for idx, row in tqdm(video_info.iterrows(), total=len(video_info)):
            vpath, vlen = row
            if vlen - self.num_seq*self.seq_len*self.downsample <= 10:
                drop_idx.append(idx)
        self.video_info = video_info.drop(drop_idx, axis=0)
        print('#videos kept: ' + str(len(self.video_info)))

        if mode == 'val':
            self.video_info = self.video_info.sample(frac=1, random_state=666)
        if self.unit_test:
            self.video_info = self.video_info.sample(32, random_state=666)

        print('construct windows of stride 1 from each video ...')
        self.window_info = pd.DataFrame(columns=['vpath', 'vlen', 'window_idx',
                                                 'all_window_seq_idx_block',
                                                 'current_frame_idx'])

        pkl_dir = self.pkl_folder_name
        os.makedirs(pkl_dir, exist_ok=True)

        pkl_name = os.path.join(pkl_dir, "window_lists.pkl")
        if os.path.exists(pkl_name):
            print("skip constructing windows... use pre-computed one...")
            with open(pkl_name, "rb") as f:
                # ← 修复：latin1
                self.window_info = pickle.load(f, encoding='latin1')
        else:
            ct_window_info_slice = 0
            v_idx_startfrom1 = 0
            for _, (vpath, vlen) in tqdm(self.video_info.iterrows(), total=len(self.video_info)):
                v_idx_startfrom1 += 1
                if v_idx_startfrom1 == 1000:
                    v_idx_startfrom1 = 0
                    with open(pkl_name + str(ct_window_info_slice), "wb") as f:
                        pickle.dump(self.window_info, f)
                    print("+1 to ct_window_info_slice:", ct_window_info_slice)
                    ct_window_info_slice += 1
                    self.window_info = pd.DataFrame(columns=['vpath', 'vlen', 'window_idx',
                                                             'all_window_seq_idx_block',
                                                             'current_frame_idx'])
                if vlen - self.num_seq*self.seq_len*self.downsample <= 0:
                    print("vlen:", vlen)
                    print("need:", self.num_seq*self.seq_len*self.downsample)
                    continue
                n_window = int(vlen/self.downsample) - (1+1)*self.seq_len + 1
                all_window_seq_idx_block = np.zeros((n_window, self.num_seq, self.seq_len))
                for window_idx in range(n_window):
                    start_zeropad = self.num_seq - 1 - self.pred_step
                    window_start_frame_idx = window_idx*self.downsample - start_zeropad*self.seq_len*self.downsample
                    seq_idx = np.expand_dims(np.arange(self.num_seq), -1)*self.downsample*self.seq_len + window_start_frame_idx
                    tmp = seq_idx + np.expand_dims(np.arange(self.seq_len),0)*self.downsample
                    tmp[tmp<0] = 0
                    tmp[tmp>vlen-1] = vlen-1
                    all_window_seq_idx_block[window_idx] = tmp
                    self.window_info.loc[len(self.window_info)] = [
                        vpath, vlen, window_idx,
                        all_window_seq_idx_block[window_idx],
                        all_window_seq_idx_block[window_idx][self.num_seq-self.pred_step-1][0]
                    ]
            # merge slices
            merge = []
            for i in range(ct_window_info_slice):
                with open(pkl_name+str(i), "rb") as f:
                    w = pickle.load(f, encoding='latin1')  # ← 修复
                    merge.append(w)
                os.remove(pkl_name+str(i))
            merge.append(self.window_info)
            self.window_info = pd.concat(merge, ignore_index=True)
            print("len(self.window_info):", len(self.window_info))
            with open(pkl_name, "wb") as f:
                pickle.dump(self.window_info, f)

    def __getitem__(self, index):
        vpath, vlen, window_idx, idx_block, current_frame_idx = self.window_info.iloc[index]
        if idx_block is None: print(vpath)
        assert idx_block.shape == (self.num_seq, self.seq_len)
        idx_block = idx_block.reshape(self.num_seq*self.seq_len)

        seq = [pil_loader(os.path.join(vpath, 'image_%05d.jpg' % (i+1))) for i in idx_block]
        t_seq = self.transform(seq)
        (C, H, W) = t_seq[0].size()
        t_seq = torch.stack(t_seq, 0)
        t_seq = t_seq.view(self.num_seq, self.seq_len, C, H, W).transpose(1,2)

        videoid = vpath.split('/')[-1][:11]
        return t_seq, videoid, vlen, window_idx, current_frame_idx

    def __len__(self):
        return len(self.window_info)

# ------------------------- Inference main -------------------------
import argparse
import matplotlib
matplotlib.use('Agg')
import torchvision
from augmentation import *
from utils import AverageMeter, save_checkpoint, denorm, calc_topk_accuracy

from tensorboardX import SummaryWriter
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import torchvision.utils as vutils

sys.path.append('../utils')
sys.path.append('../backbone')
from resnet_2d3d import neq_load_customized
from model_3d import DPC_RNN_Infer_Pred_Error, ResNet50_BeforePool_Feature_Extractor, ResNet18_BeforePool_Feature_Extractor, RBG_Extractor

torch.backends.cudnn.benchmark = True

parser = argparse.ArgumentParser()
parser.add_argument('--net', default='resnet18', type=str)
parser.add_argument('--model', default='dpc-rnn', type=str)
parser.add_argument('--dataset', default='k400', type=str)
parser.add_argument('--pred_task', default='dpc_rnn_val', type=str)
parser.add_argument('--pkl_folder_name', default='../../data/exp_k400', type=str)
parser.add_argument('--seq_len', default=5, type=int)
parser.add_argument('--num_seq', default=8, type=int)
parser.add_argument('--pred_step', default=3, type=int)
parser.add_argument('--ds', default=3, type=int)
parser.add_argument('--batch_size', default=4, type=int)
parser.add_argument('--lr', default=1e-3, type=float)
parser.add_argument('--wd', default=1e-5, type=float)
parser.add_argument('--resume', default='', type=str)
parser.add_argument('--pretrain', default='', type=str)
parser.add_argument('--epochs', default=10, type=int)
parser.add_argument('--start-epoch', default=0, type=int)
parser.add_argument('--gpu', default='0', type=str)
parser.add_argument('--print_freq', default=5, type=int)
parser.add_argument('--reset_lr', action='store_true')
parser.add_argument('--prefix', default='tmp', type=str)
parser.add_argument('--img_dim', default=224, type=int)
parser.add_argument('--unit_test', default=0, type=int)
parser.add_argument('--mode', default='val', type=str)
parser.add_argument('--modality', default='rgb', type=str)
parser.add_argument('--which_split', default=1, type=int)

def main():
    torch.manual_seed(0); np.random.seed(0)
    global args; args = parser.parse_args()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    global cuda; cuda = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 输出目录
    global pkl_folder
    if args.pkl_folder_name != '':
        pkl_folder = os.path.join(args.pkl_folder_name, args.pred_task, 'pred_err')
    else:
        base = '../../data/exp_k400' if args.dataset=='k400' else '../../data/exp_TAPOS'
        pkl_folder = os.path.join(base, args.pred_task, 'pred_err')
        args.pkl_folder_name = os.path.dirname(pkl_folder)
    os.makedirs(pkl_folder, exist_ok=True)

    # 模型
    if args.model == 'dpc-rnn':
        model = DPC_RNN_Infer_Pred_Error(sample_size=args.img_dim,
                                         num_seq=args.num_seq, seq_len=args.seq_len,
                                         network=args.net, pred_step=args.pred_step)
    elif args.model == 'resnet50-beforepool':
        model = ResNet50_BeforePool_Feature_Extractor(pretrained=True,
                                                      num_seq=args.num_seq, seq_len=args.seq_len,
                                                      pred_step=args.pred_step)
    elif args.model == 'resnet18-beforepool':
        model = ResNet18_BeforePool_Feature_Extractor(pretrained=True,
                                                      num_seq=args.num_seq, seq_len=args.seq_len,
                                                      pred_step=args.pred_step)
    elif args.model == 'rgb-avg-temporal':
        model = RBG_Extractor(pretrained=True, num_seq=args.num_seq,
                              seq_len=args.seq_len, pred_step=args.pred_step)
    else:
        raise ValueError('wrong model!')

    model = torch.nn.DataParallel(model).to(cuda)
    global criterion; criterion = torch.nn.MSELoss(reduction='none')

    print('\n===========No grad for all layers============')
    for name, param in model.named_parameters():
        param.requires_grad = False
        print(name, param.requires_grad)
    print('==============================================\n')

    if args.pretrain and os.path.isfile(args.pretrain):
        print(f"=> loading pretrained checkpoint '{args.pretrain}'")
        checkpoint = torch.load(args.pretrain, map_location='cpu')
        model = neq_load_customized(model, checkpoint['state_dict'])
        print(f"=> loaded pretrained (epoch {checkpoint.get('epoch', 'NA')})")
    else:
        if args.pretrain:
            print(f"=> no checkpoint found at '{args.pretrain}' (will run with random weights)")

    # 数据
    transform = transforms.Compose([Scale(size=(args.img_dim,args.img_dim)), ToTensor(), Normalize()])
    val_loader = get_data(transform, args.mode, args)

    validate(val_loader, model, pkl_folder)

def validate(data_loader, model, pkl_folder):
    model.eval()
    with torch.no_grad():
        for idx, (input_seq, video_id, vlen, window_idx, current_frame_idx) in tqdm(enumerate(data_loader), total=len(data_loader)):
            input_seq = input_seq.to(model.device if hasattr(model, 'device') else 'cuda')
            B = input_seq.size(0)
            target_, pred_ = model(input_seq)  # [B, pred_step, D, H, W]
            loss = criterion(pred_, target_).cpu()  # [B, pred_step, D, H, W]
            for i in range(B):
                err_dict = {
                    'pred_err': loss[i].numpy(),
                    'video_id': video_id[i],
                    'vlen': int(vlen[i].cpu().numpy()) if torch.is_tensor(vlen[i]) else int(vlen[i]),
                    'window_idx': int(window_idx[i].cpu().numpy()) if torch.is_tensor(window_idx[i]) else int(window_idx[i]),
                    'current_frame_idx': int(current_frame_idx[i].cpu().numpy()) if torch.is_tensor(current_frame_idx[i]) else int(current_frame_idx[i]),
                }
                fn = os.path.join(pkl_folder, f"err_vid_{video_id[i]}_currentframe_{str(err_dict['current_frame_idx']).zfill(6)}.pkl")
                with open(fn, "wb") as f:
                    pickle.dump(err_dict, f)
    print(f"✅ Saved prediction errors to: {pkl_folder}")

def get_data(transform, mode, args):
    print(f'Loading data for "{mode}" ...')
    use_big = args.img_dim > 140
    unit_test = bool(args.unit_test)
    if args.dataset == 'k400':
        dataset = Kinetics400_full_3d(mode=mode, transform=transform,
                                      seq_len=args.seq_len, num_seq=args.num_seq,
                                      downsample=args.ds, big=use_big, unit_test=unit_test,
                                      modality=args.modality, pkl_folder_name=args.pkl_folder_name,
                                      pred_task=args.pred_task, pred_step=args.pred_step)
    elif args.dataset == 'tapos_instances':
        dataset = TAPOS_instances_3d(mode=mode, transform=transform,
                                     seq_len=args.seq_len, num_seq=args.num_seq,
                                     downsample=args.ds, big=use_big, unit_test=unit_test,
                                     modality=args.modality, pkl_folder_name=args.pkl_folder_name,
                                     pred_task=args.pred_task, pred_step=args.pred_step)
    else:
        raise ValueError('dataset not supported')
    loader = data.DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=32, pin_memory=True)
    print(f'"{mode}" dataset size: {len(dataset)}')
    return loader

if __name__ == '__main__':
    main()
